# 2.2 kernel 的结构

## 概述

上一章我们已经跑通了 VecAdd，但还没有逐一说明这段代码是怎样工作的。本节仍以这段代码为例，先分清哪些代码在 Host 端运行、哪些代码描述 NPU 上的计算，再说明算子如何被编译和调用。最后介绍 `T.print`，用它查看 kernel 是否执行到指定位置以及当时的值。

## 学习目标

- 区分 Host 代码和 NPU kernel 计算描述。
- 解释 `@tilelang.jit`、`@T.prim_func`、`T.Kernel`、`T.Tensor/T.Buffer` 的角色。
- 使用 `T.print(value, msg=...)` 观察逻辑块信息。

## 1. Host 与 kernel

一段完整的 TileLang 算子程序同时包含 Host 代码和 kernel 描述，两部分承担的工作不同。

- **Host 代码在 CPU 上运行**，负责准备输入数据、编译 kernel、传入数据发起调用，以及比较计算结果。
- **kernel 描述 NPU 要完成的工作**，包括声明输入输出、划分逻辑任务、搬运数据、执行计算和写回结果。kernel 在定义时不会立即计算，只有经过编译并由 Host 调用后，NPU 才会真正执行。

从 kernel 描述到得到计算结果，中间有两个关键动作：**编译**把 TileLang 描述转换成设备可执行代码，**调用**则把真实 Tensor 交给已经编译好的 kernel。

<img alt="Host 与 kernel 的编译和运行过程" src="images/02.02_panorama_v4.png" style="display: block; max-width: 100%; height: auto; margin-left: 0; margin-right: auto;"/>

图注：上半部分从 TileLang 描述出发，Host 调用 JIT 入口后，编译器按选定的 target 生成对应代码并形成可调用的 `.so`；下半部分才把真实输入交给 kernel，由 NPU 计算输出，最后在 Host 侧与参考结果比较。定义 kernel 时并不会立即运行向量计算。

下面沿用上一章的 VecAdd 结构，并把编译入口写成 `@tilelang.jit` 的形式。先看注释所标出的 Host 与 kernel，这里省去了向量计算的细节：

```python
# Host 侧: JIT 编译入口: 根据 n 构造并编译 kernel
@tilelang.jit(out_idx=-1, target="ascend") # out_idx标记出输出buffer 后端选择ascendc
def make_vector_add(n):
    # Kernel 侧: @T.prim_func 标记下面的 main 是 kernel 侧的逻辑描述
    @T.prim_func
    def main(
        A: T.Buffer((n,), "float32"),  # kernel 的输入接口
        B: T.Buffer((n,), "float32"),
        C: T.Buffer((n,), "float32"),  # kernel 的输出接口
    ):
        with T.Kernel(1):                 # 划分一份逻辑任务
            # 在 NPU 上完成数据搬运、计算和写回
            ...

    return main

# Host 侧代码：在 CPU 上准备输入、取得 kernel、调用并校验
n = 1024
a = torch.randn(n, device="npu", dtype=torch.float32)
b = torch.randn(n, device="npu", dtype=torch.float32)

kernel = make_vector_add(n)  # 根据 n 即时编译，返回可调用的 kernel
out = kernel(a, b)
torch.npu.synchronize()
torch.testing.assert_close(out, a + b, rtol=0.0, atol=0.0)
```

`make_vector_add` 是 Host 侧的 Python 工厂函数，`@tilelang.jit` 把它变成即时编译入口。调用 `make_vector_add(n)` 时，参数 `n` 用来确定 shape 等编译信息，函数返回的 `main` 会被编译成可调用的 kernel；这个过程是在host侧的构造和编译计算描述，并不立即执行 `main` 中的向量计算。

由上述代码，我们可以看到 Host 与 Kernel 是如何组织从而完成一个算子的。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">程序位置</th>
<th style="text-align: left; vertical-align: top;">典型写法</th>
<th style="text-align: left; vertical-align: top;">作用</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">Host 准备数据</td>
<td style="text-align: left; vertical-align: top;"><code>torch.randn(..., device="npu")</code></td>
<td style="text-align: left; vertical-align: top;">创建输入 Tensor</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">Host 编译入口</td>
<td style="text-align: left; vertical-align: top;"><code>@tilelang.jit</code></td>
<td style="text-align: left; vertical-align: top;">根据工厂参数即时编译 kernel</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">kernel 描述</td>
<td style="text-align: left; vertical-align: top;"><code>@T.prim_func</code></td>
<td style="text-align: left; vertical-align: top;">标记需要交给编译器的计算描述</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">kernel 数据接口</td>
<td style="text-align: left; vertical-align: top;"><code>T.Tensor/T.Buffer</code></td>
<td style="text-align: left; vertical-align: top;">声明输入输出的 shape 和 dtype</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">kernel 任务范围</td>
<td style="text-align: left; vertical-align: top;"><code>T.Kernel(grid)</code></td>
<td style="text-align: left; vertical-align: top;">划分逻辑任务</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">Host 调用</td>
<td style="text-align: left; vertical-align: top;"><code>out = kernel(a, b)</code></td>
<td style="text-align: left; vertical-align: top;">传入真实数据并启动 NPU 计算</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">Host 校验</td>
<td style="text-align: left; vertical-align: top;"><code>torch.testing.assert_close(...)</code></td>
<td style="text-align: left; vertical-align: top;">将输出与参考结果完整比较</td>
</tr>
</tbody>
</table>

其中，`@tilelang.jit`、`@T.prim_func`、`T.Tensor/T.Buffer` 和 `T.Kernel` 是编写这类 kernel 时反复出现的四个基本写法。为了便于记忆，可以把它们看作一套“开头写法”：依次说明怎样编译、哪段代码是 kernel、kernel 接收什么数据，以及启动多少份逻辑任务。

**整段程序按两条线来阅读：**

- Host 侧是“准备数据 → JIT 编译 → 调用 → 校验”
- kernel 侧是“声明接口 → 划分任务 → 搬运和计算 → 写回结果”

以后看到新的 TileLang 算子，也可以先按这两条线找到程序结构，再看具体计算。

## 2. 用 `T.print` 查看 kernel 执行信息

Host 代码在 CPU 上运行，可以直接使用 Python 的 `print`。kernel 经过编译后在 NPU 上执行，普通的 Python `print` 无法观察它的执行过程。想确认 kernel 是否运行到了某个位置，或者查看某个表达式当时的值，可以使用 `T.print`。

`T.print` 是 TileLang 提供的调试原语，常用方式有两种：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">写法</th>
<th style="text-align: left; vertical-align: top;">打印效果</th>
<th style="text-align: left; vertical-align: top;">可以判断什么</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>T.print(None, msg="tilelang kernel reached")</code></td>
<td style="text-align: left; vertical-align: top;">只打印一条固定消息</td>
<td style="text-align: left; vertical-align: top;">kernel 是否执行到了这里</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>T.print(bx, msg="logical_block")</code></td>
<td style="text-align: left; vertical-align: top;">打印消息和表达式 <code>bx</code> 的值</td>
<td style="text-align: left; vertical-align: top;">当前由哪一个逻辑块执行</td>
</tr>
</tbody>
</table>

第一种写法中的 `None` 表示没有额外的数值需要打印，重点是显示 `msg`。第二种写法会把 `bx` 的当前值一起输出；在 `T.Kernel(1)` 中只有一个逻辑块，因此可以看到它的编号为 0。

```python
with T.Kernel(1) as bx:
    work = T.alloc_shared((64,), T.float32)
    T.copy(A[:], work)

    # 确认 kernel 已经执行到这里
    T.print(None, msg="tilelang kernel reached")

    # 查看当前逻辑块编号
    T.print(bx, msg="logical_block")

    T.copy(work, C[:])
```



**`T.print` 必须放在 `T.SimdVF` 外。**

这里先记住它是一个向量计算区域，具体作用将在 2.3 节结合代码介绍。多个逻辑块同时打印时，消息顺序可能交错，完整的多块调试方法留到第 6.2 节。

下面的执行单元会启动独立子进程，运行完整的打印探针。探针使用 shape `(64,)`、float32 和 `torch.arange`：除了观察两条 `T.print` 消息，还会用 `rtol=0`、`atol=0` 比较完整输出，确认加入调试语句后数据仍然正确传递。

<small>线上没有950设备时，可直接阅读本页结果；自行运行请按1.4准备环境，并从下方准备单元开始顺序执行。辅助代码仅负责启动程序与保存日志。</small>

In [1]:
from pathlib import Path
import os
import re
import subprocess
import sys
import tempfile


def run_example(script):
    """在独立进程中运行，并保留完整日志。"""
    result = subprocess.run([sys.executable, script], capture_output=True, text=True)
    log_dir = Path(os.environ.get("COURSE_OUTPUT_DIR", tempfile.gettempdir())) / "notebook_logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    log_dir.joinpath(Path(script).stem + ".log").write_text(
        result.stdout + result.stderr, encoding="utf-8"
    )
    # 页内显示程序结果；安装路径、编译过程和底层警告保留在原始日志中。
    print("\n".join(
        line for line in result.stdout.splitlines()
        if not line.startswith("Loading tilelang libs from dev root:")
        and not re.match(r"^\d{4}-\d{2}-\d{2}.*\[TileLang:", line)
    ))
    if result.returncode:
        print(result.stderr)
        result.check_returncode()


In [2]:
import subprocess, sys
run_example("src/02.02_print_probe.py")

Generated source contains:
debug_print_msg("tilelang kernel reached");
debug_print_var("logical_block", 0);
T.print runtime output is emitted above; tensor output also matches.
Verification passed!
[AIV Block 0/1] msg='tilelang kernel reached' BlockIdx=0
[AIV Block 0/1] msg='logical_block' BlockIdx=0: dtype=int value=0


## 3. 编写 kernel 时需要注意

kernel 虽然使用 Python 语法编写，但它描述的是 NPU 上的计算，因此不能直接使用任意 Python 写法。现阶段先记住三点：

- 循环使用课程中已经验证的 `range` 或 `T.serial`。
- kernel 中使用明确的索引、切片和条件，让编译器能够确定数据范围。
- `enumerate`、`zip` 和 Python 容器等复杂处理尽量放在 Host 侧，kernel 中只保留数据搬运与计算。

后续章节遇到新的语法时会结合具体算子继续说明，本节不展开完整的兼容性清单。

## 小结

现在你已经能够区分 Host 与 kernel，认出四个基本写法，并使用 `T.print` 查看 kernel 的执行信息。下一节继续了解数据在 GM、UB 和寄存器之间怎样流动。

## 课后练习

完成 range 映射、编译时参数和 T.print 题，见 [02.02_questions.txt](answer/02.02_questions.txt)；可运行的探针见 [02.02_print_probe.py](answer/02.02_print_probe.py)。